# Notebook 1 : Évaluation de MedGemma 4B avec les radiographies thoraciques (CheXpert)

Ce notebook met en place et exécute un pipeline complet d'évaluation d'un modèle
d'IA médicale (**MedGemma 4B**, Google) sur des radiographies thoraciques issues
du dataset public **CheXpert**.

**ATTENTION**, il s'agit d'un prototype strictement pédagogique : ce modèle ne fournit
aucun diagnostic médical et ne doit jamais remplacer l'avis d'un professionnel
de santé.

## Pour faire fonctionner ce notebook

1. Exporter/ouvrir ce notebook dans Google Colab.
2. Aller dans **Exécution > Modifier le type d'exécution**, et sélectionner
   **GPU T4**.
3. Avoir sous la main :
   - un **token Hugging Face** valide (avec accès accepté au modèle
     `google/medgemma-4b-it`, car c'est un modèle à accès restreint) ;
   - un fichier **`kaggle.json`** (identifiants API Kaggle, téléchargeable
     depuis son compte Kaggle) pour récupérer le dataset CheXpert.

In [1]:
import torch
print("GPU disponible :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN — active le T4 dans Exécution > Modifier le type d'exécution")

GPU disponible : True
GPU : Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Monte le Google Drive de l'utilisateur dans l'environnement Colab, au chemin `/content/drive`.
Cela permet ensuite de lire ou d'écrire des fichiers directement dans le Drive (par exemple pour sauvegarder les résultats), comme s'il s'agissait d'un disque local.

In [3]:
!pip install -q -U transformers accelerate huggingface_hub kaggle pillow pandas
!pip install -q -U "pillow>=10.4,<11" transformers accelerate huggingface_hub
import torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 24.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 w

On installe ou actualise les bibliothèques Python nécessaires au notebook :
- `transformers`, `accelerate`, `huggingface_hub` : pour charger et exécuter le modèle MedGemma.
- `kaggle` : pour télécharger le dataset CheXpert depuis Kaggle.
- `pillow`, `pandas` : pour manipuler les images et les tableaux de données.

La deuxième ligne réinstalle `pillow` avec une contrainte de version pour éviter des incompatibilités. Les messages `ERROR: pip's dependency resolver...` sont des avertissements sur des conflits de versions avec des paquets préinstallés par Colab (ex. pandas) ; ils n'empêchent pas le notebook de fonctionner.

In [4]:
from huggingface_hub import login
login()

Ouvre une invite de connexion à Hugging Face (`login()`). Il faut coller un token d'accès Hugging Face valide.
L'authentification est nécessaire pour google/medgemma-4b-it, il faut avoir accepté ses conditions d'utilisation sur Hugging Face et être connecté pour pouvoir le télécharger.

In [5]:
import time
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-4b-it"

print("Chargement de MedGemma 4B… (peut prendre 2-3 min la première fois)")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()
print(f"Modèle prêt en {time.time() - t0:.1f}s")

Chargement de MedGemma 4B… (peut prendre 2-3 min la première fois)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Modèle prêt en 152.8s


Charge le modèle **MedGemma 4B**, un modèle multimodal (image + texte) spécialisé pour des tâches médicales :
- `AutoProcessor` : charge le processeur qui prépare les entrées (texte + image) au bon format pour le modèle.
- `AutoModelForImageTextToText` : charge les poids du modèle, en `bfloat16` (demi-précision, pour économiser la mémoire GPU) et avec `device_map="auto"` pour le placer automatiquement sur le GPU.
- `.eval()` passe le modèle en mode évaluation (désactive le dropout, etc.).
- Le temps de chargement est mesuré et affiché ; la première exécution est plus longue car les poids doivent être téléchargés.

In [6]:
from google.colab import files
print("Uploade ton fichier kaggle.json :")
files.upload()  # sélectionne le kaggle.json que tu viens de télécharger

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle configuré.")

Uploade ton fichier kaggle.json :


Saving kaggle.json to kaggle.json
Kaggle configuré.


Permet d'uploader manuellement le fichier `kaggle.json` (le fichier d'identifiants de l'API Kaggle, téléchargé depuis son compte Kaggle).
Ensuite :
- crée le dossier `~/.kaggle` s'il n'existe pas,
- déplace le fichier uploadé à l'emplacement attendu par l'API Kaggle (`/root/.kaggle/kaggle.json`),
- restreint les droits du fichier (`chmod 600`) car Kaggle exige que ce fichier ne soit pas accessible en lecture par d'autres utilisateurs.
Cette étape configure l'authentification nécessaire pour pouvoir télécharger des datasets Kaggle par la suite.

In [7]:
!kaggle datasets list -s "chexpert" --max-size 15000000000

ref                                                    title                                           size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------  ---------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
ashery/chexpert                                        CheXpert-v1.0-small                      11496130509  2021-04-28 11:38:12.103000          26124         57  0.7647059        
mimsadiislam/chexpert                                  CheXpert                                 11505076821  2020-11-04 22:23:11.013000           4738         26  0.1764706        
willarevalo/chexpert-v10-small                         CheXpert_v1.0_small                      11505076821  2020-09-16 19:32:55.187000          15250         12  0.1764706        
amritpal333/chexpert-train-csv-modified                Chexpert Train csv modified             

Interroge l'API Kaggle pour lister les datasets publics dont le nom contient "chexpert", en filtrant sur une taille maximale de 15 Go,
ça permet de vérifier que le dataset recherché existe bien et de récupérer son identifiant exact avant de le télécharger.

In [8]:
import os
os.makedirs('/content/chexpert', exist_ok=True)

# Téléchargement (~11 Go, patiente quelques minutes)
!kaggle datasets download -d ashery/chexpert -p /content/chexpert

# Décompression
!unzip -q /content/chexpert/chexpert.zip -d /content/chexpert
print("Terminé.")

Dataset URL: https://www.kaggle.com/datasets/ashery/chexpert
License(s): CC0-1.0
100% 10.7G/10.7G [01:32<00:00, 125MB/s]

Terminé.


Télécharge et décompresse le dataset CheXpert :
- crée le dossier `/content/chexpert`,
- télécharge l'archive du dataset `ashery/chexpert` (~11 Go) via l'API Kaggle,
- décompresse le fichier `chexpert.zip` dans ce même dossier

In [9]:
import subprocess
# Cherche tous les CSV
for root, dirs, fnames in os.walk('/content/chexpert'):
    for f in fnames:
        if f.endswith('.csv'):
            print(os.path.join(root, f))

/content/chexpert/valid.csv
/content/chexpert/train.csv


Parcourt récursivement l'arborescence du dossier `/content/chexpert` (via `os.walk`) et affiche le chemin de tous les fichiers `.csv` trouvés.
Cela sert à repérer les fichiers d'annotations du dataset (comme `valid.csv`) sans avoir à connaître à l'avance la structure exacte des dossiers décompressés.

In [10]:
import pandas as pd

df = pd.read_csv('/content/chexpert/valid.csv')

print("Nombre de lignes :", len(df))
print("\nColonnes :")
for c in df.columns:
    print("  -", c)

print("\nAperçu des premières lignes :")
df.head(3)

Nombre de lignes : 234

Colonnes :
  - Path
  - Sex
  - Age
  - Frontal/Lateral
  - AP/PA
  - No Finding
  - Enlarged Cardiomediastinum
  - Cardiomegaly
  - Lung Opacity
  - Lung Lesion
  - Edema
  - Consolidation
  - Pneumonia
  - Atelectasis
  - Pneumothorax
  - Pleural Effusion
  - Pleural Other
  - Fracture
  - Support Devices

Aperçu des premières lignes :


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/valid/patient64541/study1/...,Male,73,Frontal,AP,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CheXpert-v1.0-small/valid/patient64542/study1/...,Male,70,Frontal,PA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,CheXpert-v1.0-small/valid/patient64542/study1/...,Male,70,Lateral,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Charge le fichier `valid.csv` de CheXpert dans un DataFrame pandas.
Affiche :
- le nombre total de lignes (une ligne = une radiographie annotée),
- la liste des colonnes disponibles (chemin de l'image, orientation, différentes pathologies possibles, etc.),
- les 3 premières lignes pour visualiser la structure des données.

In [11]:
import os

CHEXPERT_ROOT = '/content/chexpert'

def to_abs_path(p):
    parts = p.split('/')
    if parts[0] == 'CheXpert-v1.0-small':
        parts = parts[1:]
    return os.path.join(CHEXPERT_ROOT, *parts)

def map_to_class(row):
    if row.get('No Finding') == 1.0:
        return 'normal'
    if row.get('Lung Opacity') == 1.0:
        return 'suspected_opacity'
    return 'uncertain'

# Frontales uniquement + colonnes calculées
frontal = df[df['Frontal/Lateral'] == 'Frontal'].copy()
frontal['true_class'] = frontal.apply(map_to_class, axis=1)
frontal['abs_path'] = frontal['Path'].apply(to_abs_path)

# Échantillon équilibré : on tire les index par classe, puis on récupère les lignes
N_PER_CLASS = 10
idx = []
for cls, group in frontal.groupby('true_class'):
    idx += list(group.sample(min(len(group), N_PER_CLASS), random_state=42).index)

sample = frontal.loc[idx].reset_index(drop=True)

print("Répartition :")
print(sample['true_class'].value_counts())
print("\nVérif chemins (doivent tous être True) :")
print(sample['abs_path'].apply(os.path.exists).value_counts())
print("\nExemple :", sample['abs_path'].iloc[0])
sample[['abs_path', 'true_class']].head()

Répartition :
true_class
normal               10
suspected_opacity    10
uncertain            10
Name: count, dtype: int64

Vérif chemins (doivent tous être True) :
abs_path
True    30
Name: count, dtype: int64

Exemple : /content/chexpert/valid/patient64576/study1/view1_frontal.jpg


,abs_path,true_class
0,/content/chexpert/valid/patient64576/study1/vi...,normal
1,/content/chexpert/valid/patient64599/study1/vi...,normal
2,/content/chexpert/valid/patient64544/study1/vi...,normal
3,/content/chexpert/valid/patient64721/study1/vi...,normal
4,/content/chexpert/valid/patient64588/study1/vi...,normal


On construit l'échantillon de 30 images qui servira aux tests, en plusieurs étapes :
- `to_abs_path` : convertit les chemins relatifs du CSV (qui commencent par `CheXpert-v1.0-small/...`) en chemins absolus valides sur le disque de Colab.
- `map_to_class` : simplifie les nombreuses colonnes de pathologies du CSV en 3 classes : `normal` (No Finding), `suspected_opacity` (Lung Opacity), ou `uncertain` (tout le reste).
- On filtre pour ne garder que les radios **frontales** (`Frontal/Lateral == 'Frontal'`), plus faciles à interpréter.
- On tire ensuite un échantillon équilibré : jusqu'à 10 images par classe (`N_PER_CLASS = 10`), avec une graine aléatoire fixe (`random_state=42`) pour que le tirage soit reproductible.
- Enfin, on vérifie que tous les chemins d'images existent bien sur le disque, et on affiche la répartition finale par classe ainsi qu'un exemple de chemin.

In [12]:
BASELINE_PROMPT = """You are an educational radiology assistant for engineering students.
You are not a clinician and you must not provide a definitive diagnosis.

Analyze the provided frontal chest X-ray for a simple educational task:
normal vs suspected lung opacity/pneumonia-related abnormality vs uncertain.

Return only valid JSON with this schema:
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["short observation 1", "short observation 2"],
  "justification": "2 to 4 cautious sentences, evidence-based",
  "limitations": ["possible limitation 1", "possible limitation 2"],
  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."
}

Rules:
- Do not invent patient history.
- Do not mention findings that are not visible.
- Use "uncertain" when image quality is poor or evidence is weak.
- Avoid definitive clinical diagnosis.
- Keep the response concise.
"""
print("Prompt baseline défini.")

Prompt baseline défini.


Définit le **prompt "baseline"** (`BASELINE_PROMPT`) : les instructions textuelles envoyées au modèle avec chaque image.
Ce prompt demande au modèle de se comporter comme un assistant pédagogique prudent (pas de diagnostic définitif) et de répondre **uniquement** avec un objet JSON structuré contenant : la qualité de l'image, la classe prédite, un score de confiance, des observations visuelles, une justification, des limites, et un avertissement standard.

In [13]:
import time
from PIL import Image

# On prend la première image de l'échantillon
test_path = sample['abs_path'].iloc[0]
test_truth = sample['true_class'].iloc[0]
image = Image.open(test_path).convert('RGB')
print(f"Image : {test_path}")
print(f"Vérité terrain : {test_truth}\n")

messages = [
    {"role": "system", "content": [{"type": "text",
     "text": "Tu es un assistant pédagogique prudent. Tu n'établis jamais de diagnostic."}]},
    {"role": "user", "content": [
        {"type": "text", "text": BASELINE_PROMPT},
        {"type": "image", "image": image}]},
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]
t0 = time.time()
with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=300, do_sample=False)
decoded = processor.decode(generation[0][input_len:], skip_special_tokens=True)

print(f"Temps d'inférence : {time.time() - t0:.1f}s\n")
print("=== RÉPONSE BRUTE DU MODÈLE ===")
print(decoded)

Image : /content/chexpert/valid/patient64576/study1/view1_frontal.jpg
Vérité terrain : normal



[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


Temps d'inférence : 18.9s

=== RÉPONSE BRUTE DU MODÈLE ===
```json
{
  "image_quality": "good",
  "predicted_class": "normal",
  "confidence": 0.95,
  "visual_evidence": [
    "The lungs appear clear with no obvious consolidation or infiltrates.",
    "The heart size is within normal limits."
  ],
  "justification": "The image shows a clear view of the chest with no obvious signs of pneumonia or other lung abnormalities. The heart size appears normal.",
  "limitations": [
    "The image is a single frontal view, which limits the assessment of the lung fields.",
    "The patient's position and the presence of medical devices may obscure some details."
  ],
  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."
}
```


Fait un **premier test d'inférence** sur la toute première image de l'échantillon :
- charge l'image avec PIL et l'affiche avec sa vérité terrain (`true_class`),
- construit un message multimodal (system + texte du prompt baseline + image) au format attendu par le modèle,
- transforme ce message en tenseurs d'entrée via `processor.apply_chat_template`,
- lance la génération du modèle (`model.generate`, 300 tokens max, génération déterministe avec `do_sample=False`),
- décode la réponse générée et affiche le temps d'inférence ainsi que la réponse brute du modèle (censée être du JSON).


In [14]:
import time

for i in [1, 2]:  # on saute la 0 déjà testée
    image = Image.open(sample['abs_path'].iloc[i]).convert('RGB')
    messages = [
        {"role": "system", "content": [{"type": "text",
         "text": "Tu es un assistant pédagogique prudent. Tu n'établis jamais de diagnostic."}]},
        {"role": "user", "content": [
            {"type": "text", "text": BASELINE_PROMPT},
            {"type": "image", "image": image}]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    input_len = inputs["input_ids"].shape[-1]
    t0 = time.time()
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    dt = time.time() - t0
    print(f"Image {i} : {dt:.1f}s")

Image 1 : 19.4s
Image 2 : 19.7s


Répète le même test d'inférence que la cellule précédente, mais pour les images d'index 1 et 2 de l'échantillon (la première ayant déjà été testée).
Elle ne fait qu'afficher le temps d'inférence pour chaque image (`dt`), sans afficher la réponse complète du modèle : l'objectif ici est surtout de mesurer la vitesse/stabilité du pipeline sur plusieurs images avant de lancer la boucle complète.

In [15]:
print(torch.cuda.get_device_name(0))

Tesla T4


In [16]:
import json, re

def extract_json(text):
    """Extrait le premier objet JSON d'une réponse modèle, même entouré de ```json```."""
    # Retire les fences Markdown ```json ... ```
    cleaned = re.sub(r"```(?:json)?", "", text).strip().strip("`").strip()
    # Cherche le premier bloc { ... }
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError(f"Pas de JSON trouvé : {text[:150]}")
    return json.loads(match.group(0))

def normalize_result(result, prompt_variant):
    """Filet de sécurité : applique la règle d'incertitude et garantit les champs."""
    valid = {"normal", "suspected_opacity", "uncertain"}
    if result.get("predicted_class") not in valid:
        result["predicted_class"] = "uncertain"
    try:
        conf = float(result.get("confidence", 0))
    except (TypeError, ValueError):
        conf = 0.0
    conf = max(0.0, min(1.0, conf))
    result["confidence"] = conf
    # Règle dure du prompt improved : conf < 0.60 -> uncertain
    if conf < 0.60 and result["predicted_class"] != "uncertain":
        result["predicted_class"] = "uncertain"
    return result

print("Fonctions d'extraction prêtes.")

Fonctions d'extraction prêtes.


Définit deux fonctions utilitaires essentielles pour exploiter les réponses du modèle de façon fiable :
- `extract_json(text)` : extrait le premier objet JSON valide contenu dans la réponse textuelle du modèle, même si celle-ci est entourée de balises Markdown (```json ... ```). Lève une erreur explicite si aucun JSON n'est trouvé.
- `normalize_result(result, prompt_variant)` : un "filet de sécurité" qui garantit que le résultat est exploitable : force `predicted_class` à une valeur valide, s'assure que `confidence` est un nombre entre 0 et 1, et applique la règle métier « si la confiance est inférieure à 0.60, la classe prédite devient `uncertain` ».

In [17]:
def run_inference(image, prompt):
    messages = [
        {"role": "system", "content": [{"type": "text",
         "text": "Tu es un assistant pédagogique prudent. Tu n'établis jamais de diagnostic."}]},
        {"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": image}]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=220, do_sample=False)
    return processor.decode(gen[0][input_len:], skip_special_tokens=True)

print("Fonction d'inférence prête.")

Fonction d'inférence prête.


Définit la fonction `run_inference(image, prompt)`, qui encapsule toute la logique d'inférence testée dans les cellules précédentes (construction du message, tokenisation, génération, décodage) en une seule fonction réutilisable.
Elle prend en entrée une image et un prompt, et renvoie directement le texte généré par le modèle. Elle sera appelée en boucle sur les 30 images et sur les 2 variantes de prompt dans la cellule suivante.

In [18]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/assistant-radio'
os.makedirs(WORK_DIR, exist_ok=True)
print("Résultats sauvegardés dans :", WORK_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Résultats sauvegardés dans : /content/drive/MyDrive/assistant-radio



Remonte le Google Drive (au cas où la session aurait été redémarrée entre-temps) et crée un dossier de travail dédié : `/content/drive/MyDrive/assistant-radio`.
C'est dans ce dossier que seront écrits les fichiers de résultats (CSV) générés plus loin, afin qu'ils persistent au-delà de la session Colab.

In [19]:
IMPROVED_PROMPT = """You are an educational radiology assistant for engineering students.
You are not a clinician and you must not provide a definitive diagnosis.

Analyze the frontal chest X-ray under strict uncertainty rules.
Before deciding, explicitly check whether the apparent abnormality could be due to projection, rotation, poor exposure, low inspiration or overlapping anatomy. If so, reduce confidence and consider "uncertain".

Return only valid JSON with this schema:
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["short factual observation"],
  "justification": "2 to 4 cautious sentences linked to visible evidence",
  "limitations": ["image quality", "projection", "lack of clinical context"],
  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."
}

Rules:
- No patient history invention.
- No diagnosis language.
- If confidence < 0.60, predicted_class must be "uncertain".
- If JSON cannot be guaranteed, return "uncertain"."""
print("Prompt improved défini.")

Prompt improved défini.


Définit le **prompt "improved"**, une version plus stricte du prompt baseline.
Par rapport au baseline, il ajoute :
- une consigne explicite de vérifier si l'anomalie apparente pourrait être due à un artefact technique (projection, rotation, mauvaise exposition, inspiration insuffisante, superposition anatomique), auquel cas il faut réduire la confiance ou répondre `uncertain`,
- une règle dure : si la confiance est inférieure à 0.60, la classe prédite doit être `uncertain`.
Pour rendre le modèle plus prudent quant on à de l'incertitude.

In [20]:
import time, json
import pandas as pd

PROMPTS = {"baseline": BASELINE_PROMPT, "improved": IMPROVED_PROMPT}

def list_to_str(v):
    """visual_evidence et limitations sont des listes -> on les aplatit pour le CSV."""
    if isinstance(v, list):
        return " | ".join(str(x) for x in v)
    return str(v) if v is not None else ""

for variant, prompt in PROMPTS.items():
    print(f"\n===== Variante : {variant} =====")
    rows = []
    for i, r in sample.iterrows():
        path, truth = r['abs_path'], r['true_class']
        try:
            from PIL import Image
            image = Image.open(path).convert('RGB')
            t0 = time.time()
            raw = run_inference(image, prompt)
            latency = time.time() - t0
            result = normalize_result(extract_json(raw), variant)
            json_valid = True
            error = ""
        except Exception as e:
            # Réponse inexploitable : on logue l'échec sans casser la boucle
            latency = time.time() - t0 if 't0' in dir() else 0
            result = {"image_quality": "", "predicted_class": "uncertain",
                      "confidence": 0.0, "visual_evidence": [],
                      "justification": "", "limitations": [], "warning": ""}
            json_valid = False
            error = str(e)[:120]

        rows.append({
            "filename": os.path.basename(os.path.dirname(os.path.dirname(path))) + "_" + os.path.basename(path),
            "path": path,
            "true_class": truth,
            "predicted_class": result["predicted_class"],
            "correct": int(result["predicted_class"] == truth),
            "confidence": result["confidence"],
            "image_quality": result.get("image_quality", ""),
            "visual_evidence": list_to_str(result.get("visual_evidence")),
            "justification": result.get("justification", ""),
            "limitations": list_to_str(result.get("limitations")),
            "json_valid": json_valid,
            "error": error,
            "latency_s": round(latency, 1),
        })
        status = "OK" if json_valid else "JSON FAIL"
        print(f"  [{i+1}/{len(sample)}] {truth:18s} -> {result['predicted_class']:18s} "
              f"(conf {result['confidence']:.2f}, {latency:.0f}s) {status}")

    out_df = pd.DataFrame(rows)
    out_path = os.path.join(WORK_DIR, f"results_{variant}.csv")
    out_df.to_csv(out_path, index=False)
    acc = out_df['correct'].mean()
    print(f"  -> {out_path}")
    print(f"  -> Accuracy {variant} : {acc:.3f} | JSON valides : {out_df['json_valid'].mean():.0%}")

print("\nTerminé. Deux CSV écrits dans ton Drive.")


===== Variante : baseline =====
  [1/30] normal             -> normal             (conf 0.95, 17s) OK
  [2/30] normal             -> normal             (conf 0.90, 19s) OK
  [3/30] normal             -> normal             (conf 0.95, 19s) OK
  [4/30] normal             -> normal             (conf 0.95, 18s) OK
  [5/30] normal             -> normal             (conf 0.95, 19s) OK
  [6/30] normal             -> normal             (conf 0.95, 18s) OK
  [7/30] normal             -> normal             (conf 0.95, 19s) OK
  [8/30] normal             -> suspected_opacity  (conf 0.60, 19s) OK
  [9/30] normal             -> normal             (conf 0.95, 19s) OK
  [10/30] normal             -> normal             (conf 0.95, 18s) OK
  [11/30] suspected_opacity  -> uncertain          (conf 0.20, 19s) OK
  [12/30] suspected_opacity  -> normal             (conf 0.95, 19s) OK
  [13/30] suspected_opacity  -> uncertain          (conf 0.20, 14s) OK
  [14/30] suspected_opacity  -> normal             (c

C'est la boucle principale du notebook : elle exécute l'inférence sur les **30 images de l'échantillon**, pour **chacun des deux prompts** (`baseline` et `improved`), puis sauvegarde les résultats.
Pour chaque image et chaque variante :
- charge l'image, appelle `run_inference`, mesure la latence,
- extrait et normalise le JSON de réponse via `extract_json`/`normalize_result`,
- en cas d'échec (JSON non exploitable), enregistre l'erreur sans interrompre la boucle (`json_valid=False`),
- construit une ligne de résultat contenant : nom de fichier, chemin, vérité terrain, classe prédite, exactitude (`correct`), confiance, qualité d'image, preuves visuelles, justification, limitations, validité du JSON, erreur éventuelle et latence.

À la fin de chaque variante, les résultats sont rassemblés dans un DataFrame et exportés en CSV dans le Drive (`results_baseline.csv` et `results_improved.csv`), avec l'accuracy et le taux de JSON valides affichés en console.

## Exploitation des résultats

### Résultats bruts

| Variante | Accuracy | JSON valides |
|----------|----------|--------------|
| Baseline | 0.500    | 100 %        |
| Improved | 0.400    | 100 %        |

Sur 30 cas équilibrés (10 `normal`, 10 `suspected_opacity`, 10 `uncertain`),
le prompt amélioré obtient une accuracy **inférieure** au prompt baseline.
Les deux variantes produisent 100 % de JSON valides : le contrat de sortie
est respecté dans tous les cas, aucune réponse n'a dû être rejetée au parsing.

### Interprétation : un compromis prudence / exactitude

La baisse d'accuracy de l'improved n'est pas un dysfonctionnement, c'est l'effet
direct et attendu des consignes du prompt. Le prompt amélioré impose deux choses :
une vérification explicite des artefacts d'imagerie (projection, rotation,
exposition, inspiration) et une règle dure « confiance < 0.60 -> uncertain ».
Ces consignes rendent le modèle nettement plus prudent.

L'effet se lit directement dans les prédictions sur la classe `suspected_opacity` :
le baseline prédit `suspected_opacity` à plusieurs reprises (et en classe correctement
une partie), tandis que l'improved ne prédit presque jamais cette classe,
elle choisi de la classer `normal` ou `uncertain`. Le prompt amélioré échange donc de la
capacité à détecter les opacités contre une plus grande prudence. Dans un contexte
médical, ça va dans le sens recherché d'éviter les affirmations, mais ici il fait baisser l'accuracy globale.

### Limite majeure du protocole : la classe `uncertain`

Aucune des deux variantes ne traite correctement les cas étiquetés `uncertain`.
C'est une limite assumée du protocole : la classe `uncertain` provient de notre
**mapping** des étiquettes CheXpert (tous les cas qui n'étaient ni `No Finding`
ni `Lung Opacity`), et non d'une incertitude visuelle réelle de l'image. Le modèle
n'a donc aucune raison de prédire `uncertain` sur une radiographie qu'il juge
lisible. L'accuracy sur cette classe mesure en partie un artefact de notre
construction des étiquettes, pas seulement la performance du modèle.

### Pourquoi l'accuracy globale ne suffit pas

L'accuracy agrégée (0.50 vs 0.40) masque ce qui se passe réellement par classe.
Une seule métrique ne permet pas de distinguer un modèle « moins bon » d'un modèle
« plus prudent mais moins sensible ». L'analyse fine matrice de confusion,
accuracy par classe, distribution des confidences, taux de bascule vers
`uncertain` fait l'objet du notebook 2, qui transforme ce résultat brut en
une comparaison interprétable des deux variantes.

### Conclusion

Le notebook 1 valide la chaîne complète : inférence MedGemma 4B sur radiographies
CheXpert, sortie JSON structurée et conforme (100 % valide), journalisation des
prédictions, de la vérité terrain, de la confiance et de la latence dans deux CSV.
Le résultat principal est un prompt amélioré plus prudent mais moins exact sur ce
jeu de test, qu'on va analyser en détail dans
le notebook 2.

In [21]:
import os, shutil
from google.colab import files

# Dossier de destination pour les 30 images de l'echantillon utilisees dans les traces d'execution
IMAGES_DIR = '/content/images_echantillon_30'
os.makedirs(IMAGES_DIR, exist_ok=True)

for i, row in sample.iterrows():
    src = row['abs_path']
    ext = os.path.splitext(src)[1]
    # Nom de fichier explicite : index_classe.extension (ex: 00_normal.jpg)
    dst_name = f"{i:02d}_{row['true_class']}{ext}"
    shutil.copy(src, os.path.join(IMAGES_DIR, dst_name))

print(f"{len(sample)} images copiees dans {IMAGES_DIR}")

# Compression du dossier en une seule archive .zip
zip_base = '/content/images_echantillon_30'
zip_path = zip_base + '.zip'
shutil.make_archive(zip_base, 'zip', IMAGES_DIR)
print(f"Archive creee : {zip_path}")

# Declenche le telechargement direct de l'archive vers ton ordinateur
files.download(zip_path)


30 images copiees dans /content/images_echantillon_30
Archive creee : /content/images_echantillon_30.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Crée un dossier local (`/content/images_echantillon_30`) et y copie les **30 images** de l'échantillon (`sample`) utilisées pour toutes les traces d'exécution du notebook, en les renommant `index_classe.extension` (ex : `00_normal.jpg`) pour qu'elles restent facilement identifiables.
Le dossier est ensuite compressé en une archive `.zip`, puis `files.download()` déclenche le téléchargement direct de cette archive vers ton ordinateur (une fenêtre de téléchargement s'ouvrira dans le navigateur).